In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("SHOW CATALOGS").display()

catalog
adb_retail_lakehouse_dev
samples
system


In [0]:
spark.sql("SHOW EXTERNAL LOCATIONS").display()

name,url,comment
adb_retail_lakehouse_dev,abfss://unity-catalog-storage@dbstoragenfvdrm3hvo72w.dfs.core.windows.net/7405614500839737,null
el_bronze,abfss://bronze@stretaillakehouse01.dfs.core.windows.net/,null


In [0]:
display(dbutils.fs.ls("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://bronze@stretaillakehouse01.dfs.core.windows.net/config/,config/,0,1782333890000
abfss://bronze@stretaillakehouse01.dfs.core.windows.net/customers/,customers/,0,1782333863000
abfss://bronze@stretaillakehouse01.dfs.core.windows.net/products/,products/,0,1782333905000
abfss://bronze@stretaillakehouse01.dfs.core.windows.net/sales/,sales/,0,1782333897000
abfss://bronze@stretaillakehouse01.dfs.core.windows.net/stores/,stores/,0,1782333915000


In [0]:
display(
    dbutils.fs.ls("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/customers/")
)

path,name,size,modificationTime
abfss://bronze@stretaillakehouse01.dfs.core.windows.net/customers/customers_final.csv,customers_final.csv,371129,1782333947000


In [0]:
display(
    spark.read
         .option("header", "true")
         .csv("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/customers/customers_final.csv")
)

Customer_ID,First_Name,Last_Name,Email,Country,City,Registration_Date,Customer_Segment
C00001,Chasmum,Tripathi,chasmum.tripathi@outlook.com,India,Nashik,2022-05-09,Regular
C00002,Zansi,Chander,z.chander@gmail.com,India,Bengaluru,2023-03-20,Regular
C00003,Samuel,Johal,samuel.johal@yahoo.com,India,Bhopal,2023-03-26,Regular
C00004,Laban,Savant,l.savant@hotmail.com,India,Nashik,2023-01-20,Regular
C00005,Charita,Muni,charita.muni@yahoo.com,India,Hyderabad,2022-06-22,Regular
C00006,Dayamai,Talwar,dayamai.talwar@outlook.com,India,Pune,2024-03-09,Regular
C00007,Aarna,Pandit,aarna619@outlook.com,India,Warangal,2025-10-03,Regular
C00008,Dhriti,Kari,dhriti.kari@hotmail.com,India,Pune,2023-01-18,Regular
C00009,George,Pandya,g.pandya@outlook.com,India,Bengaluru,2023-09-17,Regular
C00010,Tanish,Devan,t.devan@yahoo.com,India,Warangal,2026-04-30,Regular


In [0]:
 from pyspark.sql.types import (
   StructType,
    StructField,
    StringType,
    DateType
 )

customer_schema = StructType([
    StructField("Customer_ID", StringType(), True),
    StructField("First_Name", StringType(), True),
    StructField("Last_Name", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Registration_Date", DateType(), True),
    StructField("Customer_Segment", StringType(), True)
])

In [0]:
display(customer_schema)

StructType([StructField('Customer_ID', StringType(), True), StructField('First_Name', StringType(), True), StructField('Last_Name', StringType(), True), StructField('Email', StringType(), True), StructField('Country', StringType(), True), StructField('City', StringType(), True), StructField('Registration_Date', DateType(), True), StructField('Customer_Segment', StringType(), True)])

In [0]:
df_customers = (

    spark.read.schema(customer_schema)
    .format("csv")
    .option("header", "true")
    .load("abfss://bronze@stretaillakehouse01.dfs.core.windows.net/customers/customers_final.csv")
)

df_customers.show()
df_customers.printSchema()


+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|               Email|Country|     City|Registration_Date|Customer_Segment|
+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|     C00001|   Chasmum| Tripathi|chasmum.tripathi@...|  India|   Nashik|       2022-05-09|         Regular|
|     C00002|     Zansi|  Chander| z.chander@gmail.com|  India|Bengaluru|       2023-03-20|         Regular|
|     C00003|    Samuel|    Johal|samuel.johal@yaho...|  India|   Bhopal|       2023-03-26|         Regular|
|     C00004|     Laban|   Savant|l.savant@hotmail.com|  India|   Nashik|       2023-01-20|         Regular|
|     C00005|   Charita|     Muni|charita.muni@yaho...|  India|Hyderabad|       2022-06-22|         Regular|
|     C00006|   Dayamai|   Talwar|dayamai.talwar@ou...|  India|     Pune|       2024-03-09|         Regular|
|     C00007|     A

In [0]:
df_totalcount = df_customers.count()
df_totalcount

5000

## Data Profiling

### Null Value Analysis

In [0]:
df_nullCount = df_customers.filter(col("Email").isNull())
df_nullCount.show()

+-----------+----------+---------+-----+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|     City|Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+---------+-----------------+----------------+
|     C00169|    Rushil|   Karnik| NULL|  India|   Bhopal|       2021-09-09|         Regular|
|     C00197|    Pushti|   Parikh| NULL|  India| Warangal|       2025-07-15|         Regular|
|     C00264|    Damini|  Chaudry| NULL|  India|Hyderabad|       2023-10-15|         Regular|
|     C00421|    Aayush|    Balay| NULL|  India| Vadodara|       2023-10-19|         Regular|
|     C00435|    Panini|    Banik| NULL|  India|   Nashik|       2023-12-18|         Regular|
|     C00475|       Eta|    Amble| NULL|  India|   Bhopal|       2022-05-04|         Regular|
|     C00477|     Lekha|    Kakar| NULL|  India|   Bhopal|       2023-05-15|         Regular|
|     C00571| Bhanumati|     Kala| NULL|  India| Vadodara|  

### Duplicate Records Analysis

In [0]:
df_dupCount = (
    df_customers
        .groupBy("Customer_ID")
        .count()
        .filter(col("count") > 1)
)

df_dupCount.show()

+-----------+-----+
|Customer_ID|count|
+-----------+-----+
+-----------+-----+



## Observation

No duplicate Customer_ID records were found in the Customers dataset.
The Customer_ID column satisfies the uniqueness business rule.
No duplicate removal is required before loading data into the Silver layer.

## Business Rule Validation
### 1. No Customer_ID can be null

In [0]:
df_invalid_CustomerID = df_customers.filter(col("Customer_ID").isNull())
df_invalid_CustomerID.show()

+-----------+----------+---------+-----+-------+----+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|City|Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+----+-----------------+----------------+
+-----------+----------+---------+-----+-------+----+-----------------+----------------+



### 2. First_Name  & Last_Name can't be Null

In [0]:
df_invalid_first_name = df_customers.filter(col("First_Name").isNull())
df_invalid_first_name.show()

+-----------+----------+---------+-----+-------+----+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|City|Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+----+-----------------+----------------+
+-----------+----------+---------+-----+-------+----+-----------------+----------------+



In [0]:
df_invalid_last_name = df_customers.filter(col("Last_Name").isNull())
df_invalid_last_name.show()

+-----------+----------+---------+-----+-------+----+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|City|Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+----+-----------------+----------------+
+-----------+----------+---------+-----+-------+----+-----------------+----------------+



### 3. Customer_Segmentation Cannot be Null

In [0]:
df_invalid_customer_segmentation = df_customers.filter(col("Customer_Segment").isNull())
df_invalid_customer_segmentation.show()


+-----------+----------+---------+-----+-------+----+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|City|Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+----+-----------------+----------------+
+-----------+----------+---------+-----+-------+----+-----------------+----------------+



## Observation
### No Null Customer_Segmentation was found

### 4.Registration date cannot be greater than current_date

In [0]:
df_invalid_registration_date = df_customers.filter(df_customers.Registration_Date < current_date())
df_invalid_registration_date.show()

+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|               Email|Country|     City|Registration_Date|Customer_Segment|
+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|     C00001|   Chasmum| Tripathi|chasmum.tripathi@...|  India|   Nashik|       2022-05-09|         Regular|
|     C00002|     Zansi|  Chander| z.chander@gmail.com|  India|Bengaluru|       2023-03-20|         Regular|
|     C00003|    Samuel|    Johal|samuel.johal@yaho...|  India|   Bhopal|       2023-03-26|         Regular|
|     C00004|     Laban|   Savant|l.savant@hotmail.com|  India|   Nashik|       2023-01-20|         Regular|
|     C00005|   Charita|     Muni|charita.muni@yaho...|  India|Hyderabad|       2022-06-22|         Regular|
|     C00006|   Dayamai|   Talwar|dayamai.talwar@ou...|  India|     Pune|       2024-03-09|         Regular|
|     C00007|     A

### All registration date are prior to 30 June 2026

In [0]:
df_invalid_email = df_customers.filter(col("email").contains("@")).show()

+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|               Email|Country|     City|Registration_Date|Customer_Segment|
+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|     C00001|   Chasmum| Tripathi|chasmum.tripathi@...|  India|   Nashik|       2022-05-09|         Regular|
|     C00002|     Zansi|  Chander| z.chander@gmail.com|  India|Bengaluru|       2023-03-20|         Regular|
|     C00003|    Samuel|    Johal|samuel.johal@yaho...|  India|   Bhopal|       2023-03-26|         Regular|
|     C00004|     Laban|   Savant|l.savant@hotmail.com|  India|   Nashik|       2023-01-20|         Regular|
|     C00005|   Charita|     Muni|charita.muni@yaho...|  India|Hyderabad|       2022-06-22|         Regular|
|     C00006|   Dayamai|   Talwar|dayamai.talwar@ou...|  India|     Pune|       2024-03-09|         Regular|
|     C00007|     A

In [0]:
df_customer_invalid_email= df_customers.filter(~col("Email").contains("@"))
df_customer_invalid_email.show()

+-----------+----------+---------+-----+-------+----+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|City|Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+----+-----------------+----------------+
+-----------+----------+---------+-----+-------+----+-----------------+----------------+



In [0]:
df_trim_customers = df_customers.withColumn("First_Name", trim(col("First_Name")))\
                        .withColumn("Last_Name", trim(col("Last_Name")))\
                            .withColumn("Email", trim(col("Email")))\
                            .withColumn("Country", trim(col("Country")))\
                            .withColumn("City", trim(col("City")))\
                            .withColumn("Customer_Segment", trim(col("Customer_Segment")))

                        

df_trim_customers.show()

+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|               Email|Country|     City|Registration_Date|Customer_Segment|
+-----------+----------+---------+--------------------+-------+---------+-----------------+----------------+
|     C00001|   Chasmum| Tripathi|chasmum.tripathi@...|  India|   Nashik|       2022-05-09|         Regular|
|     C00002|     Zansi|  Chander| z.chander@gmail.com|  India|Bengaluru|       2023-03-20|         Regular|
|     C00003|    Samuel|    Johal|samuel.johal@yaho...|  India|   Bhopal|       2023-03-26|         Regular|
|     C00004|     Laban|   Savant|l.savant@hotmail.com|  India|   Nashik|       2023-01-20|         Regular|
|     C00005|   Charita|     Muni|charita.muni@yaho...|  India|Hyderabad|       2022-06-22|         Regular|
|     C00006|   Dayamai|   Talwar|dayamai.talwar@ou...|  India|     Pune|       2024-03-09|         Regular|
|     C00007|     A

### Writing the notebook to silver layer

In [0]:
(
    df_trim_customers.write
        .format("parquet")
        .mode("overwrite")
        .save("abfss://silver@stretaillakehouse01.dfs.core.windows.net/customers/")
)

In [0]:
df_silver_customers = (
    spark.read
         .format("parquet")
         .load("abfss://silver@stretaillakehouse01.dfs.core.windows.net/customers/")
)

In [0]:
df_silver_customers.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Registration_Date: date (nullable = true)
 |-- Customer_Segment: string (nullable = true)



In [0]:
df_silver_customers.count()

5000

In [0]:
display(df_silver_customers.limit(20))

Customer_ID,First_Name,Last_Name,Email,Country,City,Registration_Date,Customer_Segment
C00001,Chasmum,Tripathi,chasmum.tripathi@outlook.com,India,Nashik,2022-05-09,Regular
C00002,Zansi,Chander,z.chander@gmail.com,India,Bengaluru,2023-03-20,Regular
C00003,Samuel,Johal,samuel.johal@yahoo.com,India,Bhopal,2023-03-26,Regular
C00004,Laban,Savant,l.savant@hotmail.com,India,Nashik,2023-01-20,Regular
C00005,Charita,Muni,charita.muni@yahoo.com,India,Hyderabad,2022-06-22,Regular
C00006,Dayamai,Talwar,dayamai.talwar@outlook.com,India,Pune,2024-03-09,Regular
C00007,Aarna,Pandit,aarna619@outlook.com,India,Warangal,2025-10-03,Regular
C00008,Dhriti,Kari,dhriti.kari@hotmail.com,India,Pune,2023-01-18,Regular
C00009,George,Pandya,g.pandya@outlook.com,India,Bengaluru,2023-09-17,Regular
C00010,Tanish,Devan,t.devan@yahoo.com,India,Warangal,2026-04-30,Regular


In [0]:
df_customers.count()

5000

In [0]:
df_silver_customers.count()

5000